In [7]:
#packages
import numpy as np
import numpy.ma as ma
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import tabulate
import pickle
from recsysNN_utils import *
pd.set_option('display.precision', 1)


In [9]:
#dataset
top10_df = pd.read_csv('./data/content_top10_df.csv')
bygenre_df = pd.read_csv('./data/content_bygenre_df.csv')
top10_df

,movie id,num ratings,ave rating,title,genres
0,4993,198,4.1,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy
1,5952,188,4.0,"Lord of the Rings: The Two Towers, The",Adventure|Fantasy
2,7153,185,4.1,"Lord of the Rings: The Return of the King, The",Action|Adventure|Drama|Fantasy
3,4306,170,3.9,Shrek,Adventure|Animation|Children|Comedy|Fantasy|Ro...
4,58559,149,4.2,"Dark Knight, The",Action|Crime|Drama
5,6539,149,3.8,Pirates of the Caribbean: The Curse of the Bla...,Action|Adventure|Comedy|Fantasy
6,79132,143,4.1,Inception,Action|Crime|Drama|Mystery|Sci-Fi|Thriller
7,6377,141,4.0,Finding Nemo,Adventure|Animation|Children|Comedy
8,4886,132,3.9,"Monsters, Inc.",Adventure|Animation|Children|Comedy|Fantasy
9,7361,131,4.2,Eternal Sunshine of the Spotless Mind,Drama|Romance|Sci-Fi


In [10]:
bygenre_df

,genre,num movies,ave rating/genre,ratings per genre
0,Action,321,3.4,10377
1,Adventure,234,3.4,8785
2,Animation,76,3.6,2588
3,Children,69,3.4,2472
4,Comedy,326,3.4,8911
5,Crime,139,3.5,4671
6,Documentary,13,3.8,280
7,Drama,342,3.6,10201
8,Fantasy,124,3.4,4468
9,Horror,56,3.2,1345


In [14]:
#content-based filtering with NN
#load data set
item_train, user_train,y_train, item_features, user_features, item_vecs, movie_dict, user_to_genre = load_data()

#remove userid, rating count and ave rating during training
num_user_features = user_train.shape[1] - 3 
#remove `movie id` at training time
num_item_features = item_train.shape[1] - 1

#user genre vector start
uvs = 3

#item genre vector start
ivs = 3

#start of columns to use in training, user
u_s = 3

#start of columns to use in training, items
i_s = 1

print(f'Number of training vectors: {len(item_train)}')

Number of training vectors: 50884


In [ ]:
#pretty print a few entries in the training array - user array
pprint_train(user_train, user_features, uvs, u_s, maxcount=5)

[user id],[rating count],[rating ave],Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9


In [ ]:
#For user 2, zero entries are for the movies the user has not rated


In [16]:
#few entries fo the item array
pprint_train(item_train, item_features, ivs, i_s, maxcount=5, user=False)

[movie id],year,ave rating,Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
6874,2003,4.0,1,0,0,0,0,1,0,0,0,0,0,0,0,1
8798,2004,3.8,1,0,0,0,0,1,0,1,0,0,0,0,0,1
46970,2006,3.2,1,0,0,0,1,0,0,0,0,0,0,0,0,0
48516,2006,4.3,0,0,0,0,0,1,0,1,0,0,0,0,0,1
58559,2008,4.2,1,0,0,0,0,1,0,1,0,0,0,0,0,0


In [ ]:
#the target, y, is the movie rating given by the user - first 5 ratings by user 2
print(f'y_train[:5]: {y_train[:5]}')

y_train[:5]: [4.  3.5 4.  4.  4.5]


In [18]:
#preparing the training data using `StandardScaler` btw -1 and 1 to improve
# convergence
#scale training data
item_train_unscaled = item_train
user_train_unscaled = user_train
y_train_unscaled = y_train

#for item/movie
scalerItem = StandardScaler()
scalerItem.fit(item_train)
item_train = scalerItem.transform(item_train)

#for user
scalerUser = StandardScaler()
scalerUser.fit(user_train)
user_train = scalerUser.transform(user_train)

#target scaler
scalerTarget = MinMaxScaler((-1,1))
scalerTarget.fit(y_train.reshape(-1,1))
y_train = scalerTarget.transform(y_train.reshape(-1,1))
#ynorm_test = scalerTarget.transform(y_test.reshape(-1,1))

print(np.allclose(item_train_unscaled, scalerItem.inverse_transform(item_train)))
print(np.allclose(user_train_unscaled, scalerUser.inverse_transform(user_train)))



True
True


In [ ]:
#the scaled dataset matches the unscaled dataset

In [19]:
#split the data in to training and test sets
item_train, item_test = train_test_split(item_train, train_size=0.80, shuffle=True, random_state=1)
user_train, user_test = train_test_split(user_train, train_size=0.80, shuffle=True, random_state=1)
y_train, y_test = train_test_split(y_train, train_size=0.80, shuffle=True, random_state=1)
print(f'Movie/Item training data shape: {item_train.shape}')
print(f'Movie/Item test data shape: {item_test.shape}')

Movie/Item training data shape: (40707, 17)
Movie/Item test data shape: (10177, 17)


In [ ]:
#the scaled data has a mean of zero now(-1, 1)
import pprint


pprint_train(user_train, user_features, uvs, u_s, maxcount=5)

[user id],[rating count],[rating ave],Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
1,0,-1.0,-0.8,-0.7,0.1,-0.0,-1.2,-0.4,0.6,-0.5,-0.5,-0.1,-0.6,-0.6,-0.7,-0.7
0,1,-0.7,-0.5,-0.7,-0.1,-0.2,-0.6,-0.2,0.7,-0.5,-0.8,0.1,-0.0,-0.6,-0.5,-0.4
-1,-1,-0.2,0.3,-0.4,0.4,0.5,1.0,0.6,-1.2,-0.3,-0.6,-2.3,-0.1,0.0,0.4,-0.0
0,-1,0.6,0.5,0.5,0.2,0.6,-0.1,0.5,-1.2,0.9,1.2,-2.3,-0.1,0.0,0.2,0.3
-1,0,0.7,0.6,0.5,0.3,0.5,0.4,0.6,1.0,0.6,0.3,0.8,0.8,0.4,0.7,0.7


In [21]:
#NEURAL NETWORK CONTENT-BASED FILTERING
''' 
Use a Keras sequential model
    The first layer is a dense layer with 256 units and a relu activation.
    The second layer is a dense layer with 128 units and a relu activation.
    The third layer is a dense layer with num_outputs units and a linear or no activation.
'''
num_outputs = 32
tf.random.set_seed(1)
user_NN = tf.keras.models.Sequential([
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_outputs)
])
item_NN = tf.keras.models.Sequential([
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_outputs)
])

#create the user i/p and o/p to the base n/w
input_user = tf.keras.layers.Input(shape=(num_user_features))
vu = user_NN(input_user)
vu = tf.linalg.l2_normalize(vu, axis=1)

#create the item i/p and o/p to the base network
input_item = tf.keras.layers.Input(shape=(num_item_features))
vm = item_NN(input_item)
vu = tf.linalg.l2_normalize(vm, axis=1)

#compute the dot product of the two vectors vu and vm
output = tf.keras.layers.Dot(axes=1)([vu,vm])

#specifiy the inputs and output of the model
model = tf.keras.Model([input_user, input_item], output)

model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 16)]         0           []                               
                                                                                                  
 sequential_1 (Sequential)      (None, 32)           41376       ['input_2[0][0]']                
                                                                                                  
 tf.math.l2_normalize_1 (TFOpLa  (None, 32)          0           ['sequential_1[0][0]']           
 mbda)                                                                                            
                                                                                                  
 input_1 (InputLayer)           [(None, 14)]         0           []                           

In [22]:
#unit test
from public_tests import *
test_tower(user_NN)
test_tower(item_NN)

All tests passed!
All tests passed!


In [23]:
#using mean squared error and Adam optimizer
tf.random.set_seed(1)
cost_fn = tf.keras.losses.MeanSquaredError()
opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt,
              loss=cost_fn)

In [24]:
#fit the model
tf.random.set_seed(1)
model.fit([user_train[:, u_s:], item_train[:, i_s:]], y_train, epochs=30)

Epoch 1/30
1273/1273 [==============================] - 11s 3ms/step - loss: 0.1648
Epoch 2/30
1273/1273 [==============================] - 4s 3ms/step - loss: 0.1606
Epoch 3/30
1273/1273 [==============================] - 4s 3ms/step - loss: 0.1591
Epoch 4/30
1273/1273 [==============================] - 4s 3ms/step - loss: 0.1590
Epoch 5/30
1273/1273 [==============================] - 4s 3ms/step - loss: 0.1576
Epoch 6/30
1273/1273 [==============================] - 5s 4ms/step - loss: 0.1573
Epoch 7/30
1273/1273 [==============================] - 5s 4ms/step - loss: 0.1575
Epoch 8/30
1273/1273 [==============================] - 5s 4ms/step - loss: 0.1569
Epoch 9/30
1273/1273 [==============================] - 6s 5ms/step - loss: 0.1556
Epoch 10/30
1273/1273 [==============================] - 7s 6ms/step - loss: 0.1566
Epoch 11/30
1273/1273 [==============================] - 5s 4ms/step - loss: 0.1560
Epoch 12/30
1273/1273 [==============================] - 6s 4ms/step - loss: 0.1560


In [26]:
#evalate the model to determine loss on the test data
model.evaluate([user_test[:, u_s:], item_test[:, i_s:]], y_test)

319/319 [==============================] - 1s 2ms/step - loss: 0.1493


0.1493229866027832

In [ ]:
# The test data loss ~0.14 is comparable (not far-off) to  
#  the training set loss ~0.15

In [28]:
#PREDICTIONS
#Prediction for  anew user
new_user_id = 5000
new_rating_ave = 0.0
new_action = 0.0
new_adventure = 5.0
new_animation = 0.0
new_childrens = 0.0
new_comedy = 0.0
new_crime = 0.0
new_documentary = 0.0
new_drama = 0.0
new_fantasy = 5.0
new_horror = 0.0
new_mystery = 0.0
new_romance = 0.0
new_scifi = 0.0
new_thriller = 0.0
new_rating_count = 3

user_vec = np.array([[new_user_id, new_rating_count, new_rating_ave,
                      new_action, new_adventure, new_animation, new_childrens,
                      new_comedy, new_crime, new_documentary,
                      new_drama, new_fantasy, new_horror, new_mystery,
                      new_romance, new_scifi, new_thriller]])

In [32]:
#New user enjoys adventure and fantasy genres. Let's find top-rated movies for the new user
#generate and replicate the user vector to match the number movies in the dataset
user_vecs = gen_user_vecs(user_vec, len(item_vecs))

#scale our user and item vectors
suser_vecs = scalerUser.transform(user_vecs)
sitem_vecs = scalerItem.transform(item_vecs)

#make a prediction
y_p = model.predict([suser_vecs[:, u_s:], sitem_vecs[:, i_s:]])

#unscale y prediction
y_pu = scalerTarget.inverse_transform(y_p)

#sort the results, highest prediction first
sorted_index = np.argsort(-y_pu, axis=0).reshape(-1).tolist() #negate to get the largest rating first
sorted_ypu = y_pu[sorted_index]
sorted_items = item_vecs[sorted_index]  #using unscaled vectors for display

print_pred_movies(sorted_ypu, sorted_items, movie_dict, maxcount=10)


27/27 [==============================] - 0s 2ms/step


y_p,movie id,rating ave,title,genres
4.3,168252,4.3,Logan (2017),Action|Sci-Fi
4.3,55721,4.3,Elite Squad (Tropa de Elite) (2007),Action|Crime|Drama|Thriller
4.3,96829,4.3,"Hunt, The (Jagten) (2012)",Drama
4.3,92535,4.3,Louis C.K.: Live at the Beacon Theater (2011),Comedy
4.2,48516,4.3,"Departed, The (2006)",Crime|Drama|Thriller
4.2,58559,4.2,"Dark Knight, The (2008)",Action|Crime|Drama
4.2,7156,4.3,"Fog of War: Eleven Lessons from the Life of Robert S. McNamara, The (2003)",Documentary
4.2,80906,4.3,Inside Job (2010),Documentary
4.2,8014,4.2,"Spring, Summer, Fall, Winter... and Spring (Bom yeoreum gaeul gyeoul geurigo bom) (2003)",Drama
4.2,116897,4.2,Wild Tales (2014),Comedy|Drama|Thriller


In [35]:
#prediction for an existing user
uid = 2

#form a set of user vectors. This is the same vector, transformed and repeated
user_vecs, y_vecs = get_user_vecs(uid, user_train_unscaled, item_vecs, user_to_genre)

#scale our user and item vectors
suser_vecs = scalerUser.transform(user_vecs)
sitem_vecs = scalerItem.transform(item_vecs)

#make a prediction
y_p = model.predict([suser_vecs[:, u_s:], sitem_vecs[:, i_s:]])

#unscale y prediction
y_pu = scalerTarget.inverse_transform(y_p)

#sort te results, highest prediction firt
sorted_index = np.argsort(-y_pu, axis=0).reshape(-1).tolist() #negate to get the largest rating first
sorted_ypu = y_pu[sorted_index]
sorted_items = item_vecs[sorted_index] #using unscaled vectors for display
sorted_user = user_vecs[sorted_index]
sorted_y = y_vecs[sorted_index]

#print the sorted predictions for movies rated by user 2
print_existing_user(sorted_ypu, sorted_y.reshape(-1,1), sorted_user, sorted_items, ivs, uvs, movie_dict, maxcount=50)

27/27 [==============================] - 0s 3ms/step


y_p,y,user,user genre ave,movie rating ave,movie id,title,genres
4.2,4.0,2,"[4.1,4.0,3.9]",4.3,48516,"Departed, The (2006)",Crime|Drama|Thriller
4.2,4.5,2,"[4.0,4.1,4.0]",4.2,58559,"Dark Knight, The (2008)",Action|Crime|Drama
4.2,5.0,2,[4.0],4.3,80906,Inside Job (2010),Documentary
4.1,4.5,2,"[4.0,4.0]",4.1,68157,Inglourious Basterds (2009),Action|Drama
4.1,4.0,2,"[4.0,4.1,4.0,4.0,3.9,3.9]",4.1,79132,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller
4.0,4.0,2,[4.0],4.0,112552,Whiplash (2014),Drama
4.0,3.0,2,[3.9],4.0,109487,Interstellar (2014),Sci-Fi
4.0,3.5,2,"[4.0,4.2,4.1]",4.0,91529,"Dark Knight Rises, The (2012)",Action|Adventure|Crime
4.0,4.0,2,"[4.0,4.0,3.9]",4.0,74458,Shutter Island (2010),Drama|Mystery|Thriller
4.0,3.0,2,"[4.0,4.0]",4.0,77455,Exit Through the Gift Shop (2010),Comedy|Documentary


In [ ]:
#FINDING SIMILAR 
#Version1 - using for loop
def sq_dist(a,b):
    ''' 
    Returns the squared distance between two vectors

    Args:
        a (ndarray (n,)) : vector with n features
        b (ndarray (n,)) : vector with n features
    Returns:
        d (float) : distance
    '''
    a = np.asarray(a).flatten() #ensures inputs are 1D vectors to avoid shape-related issues
    b = np.asarray(b).flatten()
    d = 0.0
    for i in  range(len(a)):
        diff = a[i] - b[i]
        d += diff **2
    return d



In [ ]:
#Version 2 - using vectorized array
def sq_dist(a,b):
    ''' 
    Returns the squared distance between two vectors

    Args:
        a (ndarray (n,)) : vector with n features
        b (ndarray (n,)) : vector with n features
    Returns:
        d (float) : distance
    '''
    a = np.array(a)
    b = np.array(b)
    d = np.sum((a - b) ** 2)
    return d

In [43]:
#implement the above code
a1 = np.array([1.0, 2.0, 3.0]); b1 = np.array([1.0, 2.0, 3.0])
a2 = np.array([1.1, 2.1, 3.1]); b2 = np.array([1.0, 2.0, 3.0])
a3 = np.array([0, 1, 0]);       b3 = np.array([1, 0, 0])
print(f"squared distance between a1 and b1: {sq_dist(a1, b1):0.3f}")
print(f"squared distance between a2 and b2: {sq_dist(a2, b2):0.3f}")
print(f"squared distance between a3 and b3: {sq_dist(a3, b3):0.3f}")

squared distance between a1 and b1: 0.000
squared distance between a2 and b2: 0.030
squared distance between a3 and b3: 2.000


In [41]:
#unit test
test_sq_dist(sq_dist)

All tests passed!


In [44]:
#vm can be pre-computed and reused for new recommendations
input_item_m = tf.keras.layers.Input(shape=(num_item_features)) #input later
vm_m  = item_NN(input_item_m) #use the trained iten_NN
vm_m = tf.linalg.l2_normalize(vm_m, axis=1) #incorporate normalization as was done in the original model
model_m = tf.keras.Model(input_item_m, vm_m)
model_m.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 16)]              0         
                                                                 
 sequential_1 (Sequential)   (None, 32)                41376     
                                                                 
 tf.math.l2_normalize_2 (TFO  (None, 32)               0         
 pLambda)                                                        
                                                                 
Total params: 41,376
Trainable params: 41,376
Non-trainable params: 0
_________________________________________________________________


In [ ]:
#create a set of moviefeatures using the above model to predict usin a set
# of item/movie vectors as input
#The result of the prediction is a 32 entry feature vector for each movie
scaled_item_vecs = scalerItem.transform(item_vecs)
vms = model_m.predict(scaled_item_vecs[:, i_s:])
print(f'Size of all predicted movie feature vectors: {vms.shape}')

27/27 [==============================] - 4s 7ms/step
Size of all predicted movie feature vectors: (847, 32)


In [46]:
#Create a matrix of te squared distance btw each movie feature and all other movie feature vectors
# Use `numpy_masked_arrays` to avoid selecting the same movie
# The masked diagonal values won't be included in the computation

count = 50 #number of movies to display
dim = len(vms)
dist = np.zeros((dim,dim))

#create a 2D matrix
for i in range(dim):
    for j in range(dim):
        dist[i,j] = sq_dist(vms[i, :], vms[j, :])

m_dist = ma.masked_array(dist, mask=np.identity(dist.shape[0])) #mask the diagonal

disp = [['movie1', 'genres', 'movie2', 'genres']]
for i in range(count):
    min_idx = np.argmin(m_dist[i])
    movie1_id = int(item_vecs[i,0])
    movie2_id = int(item_vecs[min_idx, 0])
    disp.append( [movie_dict[movie1_id]['title'], movie_dict[movie1_id]['genres'],
                  movie_dict[movie2_id]['title'], movie_dict[movie2_id]['genres']]
                  )
table = tabulate.tabulate(disp, tablefmt='html', headers='firstrow')
table

movie1,genres,movie2,genres
Save the Last Dance (2001),Drama|Romance,Saving Silverman (Evil Woman) (2001),Comedy|Romance
"Wedding Planner, The (2001)",Comedy|Romance,Down with Love (2003),Comedy|Romance
Hannibal (2001),Horror|Thriller,King Arthur (2004),Action|Adventure|Drama
Saving Silverman (Evil Woman) (2001),Comedy|Romance,Save the Last Dance (2001),Drama|Romance
Down to Earth (2001),Comedy|Fantasy|Romance,Save the Last Dance (2001),Drama|Romance
"Mexican, The (2001)",Action|Comedy,Birdman: Or (The Unexpected Virtue of Ignorance) (2014),Comedy|Drama
15 Minutes (2001),Thriller,Jumper (2008),Action|Adventure|Drama|Sci-Fi|Thriller
Enemy at the Gates (2001),Drama,Ocean's Thirteen (2007),Crime|Thriller
Heartbreakers (2001),Comedy|Crime|Romance,Save the Last Dance (2001),Drama|Romance
Spy Kids (2001),Action|Adventure|Children|Comedy,In the Bedroom (2001),Drama
